# Session 4 — Your turn: Evaluation: when is a model good?

The session chose a threshold under the policy "find at least 95% of the malignant cases". Here the written policy is different, and you will apply the protocol again:
tune and choose on the validation set, lock, then open the test set **once**.

Run the cells from the top. Replace every `...` with your own code, then run the check below it. A check prints PASS or FAIL with a hint; you can run a task as often as you like.

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)              # 1 = malignant, the case we want to find
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000)).fit(X_train, y_train)
val_probability = model.predict_proba(X_val)[:, 1]
test_probability = model.predict_proba(X_test)[:, 1]          # do not look at this until task 1 tells you to
print(len(X_train), len(X_val), len(X_test))
from aij_check import check, close, summary

## Task 1: A different policy

New policy: **precision of at least 0.99 on the validation set (almost no false alarms), and among the thresholds that satisfy it, the highest recall.**
Try the thresholds `np.linspace(0.05, 0.95, 91)` on the validation set, store the best one in `threshold`, its validation precision and recall in `val_precision` and `val_recall`.
Only then open the test set once at that threshold: `test_precision` and `test_recall`.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
threshold = ...
val_precision = ...
val_recall = ...
test_precision = ...
test_recall = ...
print(threshold, val_precision, val_recall, test_precision, test_recall)

In [ ]:
check("the threshold meets the policy on the validation set", lambda: precision_score(y_val, (val_probability >= threshold).astype(int)) >= 0.99,
      "Precision on the VALIDATION set must be at least 0.99.")
check("no other threshold meets the policy with a higher recall", lambda: all(
    recall_score(y_val, (val_probability >= t).astype(int)) <= val_recall + 1e-9 for t in np.linspace(0.05, 0.95, 91)
    if (val_probability >= t).sum() > 0 and precision_score(y_val, (val_probability >= t).astype(int)) >= 0.99))
check("the validation numbers match the threshold", lambda: close(val_recall, recall_score(y_val, (val_probability >= threshold).astype(int)), 1e-9))
check("the test numbers were measured at that threshold", lambda: close(test_recall, recall_score(y_test, (test_probability >= threshold).astype(int)), 1e-9)
      and close(test_precision, precision_score(y_test, (test_probability >= threshold).astype(int)), 1e-9))
summary()

## Task 2: The four boxes by hand

A model flagged 40 tumours as malignant. Of them 38 were malignant (true positives) and 2 were benign (false positives). It missed 4 malignant tumours (false negatives) and correctly cleared 70 benign ones (true negatives).
Compute, **without scikit-learn**, `precision`, `recall`, `f1` and `accuracy`, and the number of malignant cases missed per 1,000 malignant cases, `missed_per_1000`.

In [ ]:
tp, fp, fn, tn = 38, 2, 4, 70
precision = ...
recall = ...
f1 = ...
accuracy = ...
missed_per_1000 = ...
print(precision, recall, f1, accuracy, missed_per_1000)

In [ ]:
check("precision = TP / (TP + FP)", lambda: close(precision, 38 / 40, 1e-9))
check("recall = TP / (TP + FN)", lambda: close(recall, 38 / 42, 1e-9))
check("f1 is the harmonic mean of the two", lambda: close(f1, 2 * (38 / 40) * (38 / 42) / ((38 / 40) + (38 / 42)), 1e-9))
check("accuracy counts both kinds of correct answer", lambda: close(accuracy, 108 / 114, 1e-9))
check("about 95 malignant cases missed per 1,000", lambda: close(missed_per_1000, 95.238, 0.01))
summary()